# 第三课：CNN 图像分类（可视化特征图与训练过程）

**学习目标**：用 CNN 做 MNIST 手写数字识别，可视化图片、预测、特征图、训练曲线。

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## 1. 加载并可视化 MNIST 数据

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32')/255.0
x_test = x_test.astype('float32')/255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# 展示前20张
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title('标签:%d'%y_train[i]); ax.axis('off')
plt.suptitle('MNIST 训练样本（28x28灰度图）', fontsize=14)
plt.tight_layout(); plt.show()

# 标签分布
plt.figure(figsize=(8,4))
plt.hist(y_train, bins=range(11), rwidth=0.8, align='left', color='steelblue', edgecolor='k')
plt.xticks(range(10)); plt.xlabel('数字'); plt.ylabel('样本数')
plt.title('训练集标签分布（60000张）'); plt.grid(alpha=0.3, axis='y')
plt.show()

## 2. 搭建 CNN 模型

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1), name='conv1'),
    tf.keras.layers.MaxPooling2D((2,2), name='pool1'),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', name='conv2'),
    tf.keras.layers.MaxPooling2D((2,2), name='pool2'),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', name='conv3'),
    tf.keras.layers.Flatten(name='flatten'),
    tf.keras.layers.Dense(64, activation='relu', name='dense1'),
    tf.keras.layers.Dense(10, activation='softmax', name='output')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 3. 训练并可视化过程

In [ ]:
history = model.fit(x_train, y_train, epochs=5, batch_size=128,
                     validation_split=0.1, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], 'b-o', ms=4, label='训练损失')
axes[0].plot(history.history['val_loss'], 'r-o', ms=4, label='验证损失')
axes[0].set_title('损失曲线'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['accuracy'], 'b-o', ms=4, label='训练准确率')
axes[1].plot(history.history['val_accuracy'], 'r-o', ms=4, label='验证准确率')
axes[1].set_title('准确率曲线'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print('测试集准确率: %.2f%%' % (test_acc*100))

## 4. 预测结果可视化（正确 vs 错误）

In [ ]:
preds = model.predict(x_test, verbose=0)
pred_labels = np.argmax(preds, axis=1)

# 找一些正确和错误的样本
correct = np.where(pred_labels == y_test)[0]
wrong = np.where(pred_labels != y_test)[0]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i in range(8):
    idx = correct[i]
    axes[0,i].imshow(x_test[idx].squeeze(), cmap='gray')
    axes[0,i].set_title('预测:%d\n真实:%d'%(pred_labels[idx],y_test[idx]), color='green', fontsize=9)
    axes[0,i].axis('off')
for i in range(min(8, len(wrong))):
    idx = wrong[i]
    axes[1,i].imshow(x_test[idx].squeeze(), cmap='gray')
    axes[1,i].set_title('预测:%d\n真实:%d'%(pred_labels[idx],y_test[idx]), color='red', fontsize=9)
    axes[1,i].axis('off')
axes[0,0].set_ylabel('正确预测', rotation=0, labelpad=40, fontsize=12)
axes[1,0].set_ylabel('错误预测', rotation=0, labelpad=40, fontsize=12)
plt.suptitle('预测结果展示（上：正确，下：错误）', fontsize=14)
plt.tight_layout(); plt.show()

## 5. 预测概率分布（模型有多确信？）

In [ ]:
idx = 0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(x_test[idx].squeeze(), cmap='gray')
axes[0].set_title('输入图片（真实标签:%d）'%y_test[idx]); axes[0].axis('off')
colors = ['green' if i==pred_labels[idx] else 'steelblue' for i in range(10)]
axes[1].bar(range(10), preds[idx], color=colors)
axes[1].set_xticks(range(10)); axes[1].set_xlabel('数字'); axes[1].set_ylabel('概率')
axes[1].set_title('预测概率分布（最高=预测:%d, 置信度=%.1f%%）'%(pred_labels[idx],preds[idx][pred_labels[idx]]*100))
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 6. CNN 特征图可视化（模型在看什么？）

每一层卷积核都在提取不同的特征：边缘、纹理、图案。

In [ ]:
# 构建能输出中间层的模型
layer_outputs = [model.get_layer(name).output for name in ['conv1','pool1','conv2','pool2','conv3']]
activation_model = tf.keras.Model(inputs=model.input, outputs=layer_outputs)

img = x_test[0:1]  # 选一张'7'
activations = activation_model.predict(img, verbose=0)
layer_names = ['conv1(32核)', 'pool1', 'conv2(64核)', 'pool2', 'conv3(64核)']

fig, axes = plt.subplots(5, 8, figsize=(16, 10))
for row, (name, act) in enumerate(zip(layer_names, activations)):
    for col in range(8):
        ax = axes[row, col]
        if col < act.shape[-1]:
            ax.imshow(act[0, :, :, col], cmap='viridis')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(name, rotation=0, labelpad=50, fontsize=10, va='center')
plt.suptitle('CNN 各层特征图（每行展示前8个通道）', fontsize=14)
plt.tight_layout(); plt.show()

## 7. 卷积核可视化

In [ ]:
conv1_weights = model.get_layer('conv1').get_weights()[0]
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[-1]:
        kernel = conv1_weights[:, :, 0, i]
        ax.imshow(kernel, cmap='gray')
    ax.axis('off')
plt.suptitle('第一层32个卷积核（3x3）', fontsize=14)
plt.tight_layout(); plt.show()

## 第三课小结

- CNN = 卷积层（提特征）+ 池化层（降维）+ 全连接层（分类）
- 特征图可视化能帮你理解模型学到了什么
- 训练曲线（损失/准确率）是诊断过拟合的关键

**下一课**：迁移学习，用预训练模型做你的钢坯数据。